# Practical Workflow: End-to-End NTM Analysis

This notebook provides a complete workflow for using the Neural Thermodynamic Metric to analyze molecular transformations.

## Table of Contents
1. [Setup and Dependencies](#1-setup)
2. [Loading or Training a Model](#2-loading-model)
3. [Analyzing a Molecular Pair](#3-analyzing-pair)
4. [Batch Analysis](#4-batch-analysis)
5. [Interpreting Results](#5-interpreting-results)
6. [Generating Reports](#6-generating-reports)

## 1. Setup and Dependencies <a name="1-setup"></a>

### Required Packages

```bash
pip install torch numpy rdkit plotly pandas scipy
```

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datetime import datetime

# Check dependencies
dependencies = {
    'torch': True,
    'numpy': True,
    'pandas': True
}

try:
    from rdkit import Chem
    from rdkit.Chem import Draw, AllChem, Descriptors
    dependencies['rdkit'] = True
except ImportError:
    dependencies['rdkit'] = False

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    dependencies['plotly'] = True
except ImportError:
    dependencies['plotly'] = False

print("Dependency Check")
print("=" * 30)
for pkg, available in dependencies.items():
    status = 'OK' if available else 'MISSING'
    print(f"{pkg:15} {status}")

if not all(dependencies.values()):
    print("\nInstall missing packages before proceeding.")

In [ ]:
# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Add src to path
sys.path.insert(0, '../src')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Loading or Training a Model <a name="2-loading-model"></a>

### Option A: Load Pre-trained Weights

In [ ]:
# Import the NTM model (from previous notebooks or src/)
# For this tutorial, we'll define the essential components inline

class MolecularGNN(nn.Module):
    """Simplified GNN for molecular embedding."""
    
    def __init__(self, node_dim=7, edge_dim=3, hidden_dim=64, num_layers=3):
        super().__init__()
        self.node_embed = nn.Linear(node_dim, hidden_dim)
        self.edge_embed = nn.Linear(edge_dim, hidden_dim)
        
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.LayerNorm(hidden_dim)
            ))
        
        self.output = nn.Linear(hidden_dim, hidden_dim)
    
    def forward(self, node_features, edge_index, edge_features, batch):
        x = self.node_embed(node_features)
        
        for layer in self.layers:
            # Simple message passing
            src, dst = edge_index
            messages = torch.zeros_like(x)
            messages.index_add_(0, dst, x[src])
            
            # Update
            x = layer(torch.cat([x, messages], dim=-1))
        
        # Global mean pooling
        unique_batches = batch.unique()
        pooled = torch.stack([x[batch == b].mean(dim=0) for b in unique_batches])
        
        return self.output(pooled)


class NeuralThermodynamicMetric(nn.Module):
    """Full NTM model."""
    
    def __init__(self, node_dim=7, edge_dim=3, hidden_dim=64, num_layers=3, metric_type='diagonal'):
        super().__init__()
        self.gnn = MolecularGNN(node_dim, edge_dim, hidden_dim, num_layers)
        self.hidden_dim = hidden_dim
        self.metric_type = metric_type
        
        # Metric tensor
        if metric_type == 'diagonal':
            self.log_weights = nn.Parameter(torch.zeros(hidden_dim))
        else:
            self.L_diag = nn.Parameter(torch.ones(hidden_dim))
            self.L_lower = nn.Parameter(torch.zeros(hidden_dim * (hidden_dim - 1) // 2))
        
        # Prediction head
        self.pred_head = nn.Sequential(
            nn.Linear(1 + 2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def get_metric_weights(self):
        if self.metric_type == 'diagonal':
            return torch.exp(self.log_weights)
        else:
            M = self.get_full_metric()
            return torch.diag(M)
    
    def get_full_metric(self):
        if self.metric_type == 'diagonal':
            return torch.diag(torch.exp(self.log_weights))
        else:
            L = torch.zeros(self.hidden_dim, self.hidden_dim, device=self.L_diag.device)
            L.diagonal().copy_(F.softplus(self.L_diag) + 0.1)
            idx = torch.tril_indices(self.hidden_dim, self.hidden_dim, offset=-1)
            L[idx[0], idx[1]] = self.L_lower
            return L @ L.T
    
    def encode(self, graph):
        return self.gnn(
            graph['node_features'],
            graph['edge_index'],
            graph['edge_features'],
            graph['batch']
        )
    
    def compute_distance(self, h_a, h_b):
        diff = h_b - h_a
        if self.metric_type == 'diagonal':
            weights = torch.exp(self.log_weights)
            d_sq = torch.sum(diff ** 2 * weights, dim=-1)
        else:
            M = self.get_full_metric()
            d_sq = torch.sum(diff * (diff @ M), dim=-1)
        return torch.sqrt(d_sq + 1e-8)
    
    def forward(self, graph_a, graph_b):
        h_a = self.encode(graph_a)
        h_b = self.encode(graph_b)
        
        riemannian_dist = self.compute_distance(h_a, h_b).unsqueeze(-1)
        features = torch.cat([riemannian_dist, h_b - h_a, h_a + h_b], dim=-1)
        
        return self.pred_head(features).squeeze(-1)


# Initialize model
model = NeuralThermodynamicMetric(
    node_dim=7, edge_dim=3, hidden_dim=64, 
    num_layers=3, metric_type='diagonal'
).to(DEVICE)

print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Load pre-trained weights (if available)
weights_path = "../weights/ntm_model.pt"  # Update with your path

if os.path.exists(weights_path):
    state_dict = torch.load(weights_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print(f"Loaded pre-trained weights from {weights_path}")
else:
    print("No pre-trained weights found. Using randomly initialized model.")
    print("Note: Results will not be meaningful without training.")

### Option B: Train on Your Data

If you have RBFE data with uncertainty labels:

In [ ]:
def train_model(model, train_data, val_data=None, epochs=100, lr=0.001):
    """
    Train the NTM model on RBFE data.
    
    Args:
        model: NeuralThermodynamicMetric model
        train_data: List of (graph_a, graph_b, target) tuples
        val_data: Optional validation data
        epochs: Number of training epochs
        lr: Learning rate
    
    Returns:
        Training history
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_losses = []
        
        for graph_a, graph_b, target in train_data:
            optimizer.zero_grad()
            
            pred = model(graph_a, graph_b)
            loss = F.mse_loss(pred, target)
            
            # Metric regularization
            weights = model.get_metric_weights()
            reg_loss = 0.01 * torch.mean(torch.abs(torch.log(weights + 1e-8)))
            
            total_loss = loss + reg_loss
            total_loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
        
        avg_train_loss = np.mean(train_losses)
        history['train_loss'].append(avg_train_loss)
        
        # Validation
        if val_data:
            model.eval()
            val_losses = []
            
            with torch.no_grad():
                for graph_a, graph_b, target in val_data:
                    pred = model(graph_a, graph_b)
                    loss = F.mse_loss(pred, target)
                    val_losses.append(loss.item())
            
            avg_val_loss = np.mean(val_losses)
            history['val_loss'].append(avg_val_loss)
            scheduler.step(avg_val_loss)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), 'best_model.pt')
        
        if epoch % 10 == 0:
            val_str = f", Val: {avg_val_loss:.4f}" if val_data else ""
            print(f"Epoch {epoch:3d}: Train Loss: {avg_train_loss:.4f}{val_str}")
    
    return history

print("Training function defined.")
print("Usage: history = train_model(model, train_data, val_data, epochs=100)")

## 3. Analyzing a Molecular Pair <a name="3-analyzing-pair"></a>

### Convert SMILES to Graph

In [ ]:
def mol_to_graph(smiles, device='cpu'):
    """
    Convert SMILES string to graph representation.
    """
    if not dependencies['rdkit']:
        raise ImportError("RDKit is required for SMILES conversion")
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Node features
    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization()),
            int(atom.GetIsAromatic()),
            atom.GetTotalNumHs(),
            int(atom.IsInRing())
        ]
        node_features.append(features)
    
    # Edge features and connectivity
    edge_index = []
    edge_features = []
    
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
        
        bond_feat = [
            float(bond.GetBondTypeAsDouble()),
            int(bond.GetIsConjugated()),
            int(bond.IsInRing())
        ]
        edge_features.extend([bond_feat, bond_feat])
    
    return {
        'node_features': torch.tensor(node_features, dtype=torch.float32, device=device),
        'edge_index': torch.tensor(edge_index, dtype=torch.long, device=device).T,
        'edge_features': torch.tensor(edge_features, dtype=torch.float32, device=device),
        'batch': torch.zeros(len(node_features), dtype=torch.long, device=device)
    }


def analyze_pair(model, smiles_a, smiles_b, device='cpu'):
    """
    Analyze a pair of molecules.
    
    Returns:
        dict with analysis results
    """
    model.eval()
    
    # Convert to graphs
    graph_a = mol_to_graph(smiles_a, device)
    graph_b = mol_to_graph(smiles_b, device)
    
    if graph_a is None or graph_b is None:
        return {'error': 'Invalid SMILES'}
    
    with torch.no_grad():
        # Get embeddings
        h_a = model.encode(graph_a)
        h_b = model.encode(graph_b)
        
        # Compute distances
        riemannian_dist = model.compute_distance(h_a, h_b).item()
        euclidean_dist = torch.norm(h_b - h_a).item()
        
        # Prediction
        prediction = model(graph_a, graph_b).item()
    
    # Molecular properties (for context)
    mol_a = Chem.MolFromSmiles(smiles_a)
    mol_b = Chem.MolFromSmiles(smiles_b)
    
    results = {
        'smiles_a': smiles_a,
        'smiles_b': smiles_b,
        'riemannian_distance': riemannian_dist,
        'euclidean_distance': euclidean_dist,
        'warping_factor': riemannian_dist / (euclidean_dist + 1e-8),
        'predicted_difficulty': prediction,
        'mol_a_atoms': mol_a.GetNumAtoms(),
        'mol_b_atoms': mol_b.GetNumAtoms(),
        'atom_difference': abs(mol_a.GetNumAtoms() - mol_b.GetNumAtoms()),
        'mol_a_mw': Descriptors.MolWt(mol_a),
        'mol_b_mw': Descriptors.MolWt(mol_b),
        'embedding_a': h_a.cpu().numpy(),
        'embedding_b': h_b.cpu().numpy()
    }
    
    return results


# Example analysis
if dependencies['rdkit']:
    smiles_a = "c1ccc(CC(=O)O)cc1"   # Phenylacetic acid
    smiles_b = "c1ccc(CC(=O)N)cc1"   # Phenylacetamide
    
    results = analyze_pair(model, smiles_a, smiles_b, DEVICE)
    
    print("Pair Analysis Results")
    print("=" * 50)
    print(f"Molecule A: {results['smiles_a']}")
    print(f"Molecule B: {results['smiles_b']}")
    print(f"\nDistances:")
    print(f"  Riemannian: {results['riemannian_distance']:.4f}")
    print(f"  Euclidean:  {results['euclidean_distance']:.4f}")
    print(f"  Warping:    {results['warping_factor']:.2f}x")
    print(f"\nPredicted Difficulty: {results['predicted_difficulty']:.4f}")
    print(f"\nMolecular Properties:")
    print(f"  Mol A: {results['mol_a_atoms']} atoms, MW={results['mol_a_mw']:.1f}")
    print(f"  Mol B: {results['mol_b_atoms']} atoms, MW={results['mol_b_mw']:.1f}")
else:
    print("RDKit required for this analysis.")

## 4. Batch Analysis <a name="4-batch-analysis"></a>

Analyze multiple molecular pairs efficiently:

In [ ]:
def batch_analyze(model, pairs, device='cpu'):
    """
    Analyze multiple molecular pairs.
    
    Args:
        model: NTM model
        pairs: List of (smiles_a, smiles_b) tuples
        device: Compute device
    
    Returns:
        DataFrame with results
    """
    results = []
    
    for smiles_a, smiles_b in pairs:
        result = analyze_pair(model, smiles_a, smiles_b, device)
        if 'error' not in result:
            # Remove embeddings for DataFrame
            result_clean = {k: v for k, v in result.items() 
                           if not k.startswith('embedding')}
            results.append(result_clean)
        else:
            print(f"Skipping invalid pair: {smiles_a} -> {smiles_b}")
    
    return pd.DataFrame(results)


# Example batch analysis
if dependencies['rdkit']:
    example_pairs = [
        ("c1ccc(CC(=O)O)cc1", "c1ccc(CC(=O)N)cc1"),     # COOH -> CONH2
        ("c1ccc(CC(=O)O)cc1", "c1ccc(CCC(=O)O)cc1"),    # Add CH2
        ("c1ccccc1", "c1ccc(O)cc1"),                     # Benzene -> Phenol
        ("c1ccccc1", "c1ccc(N)cc1"),                     # Benzene -> Aniline
        ("CCO", "CCCO"),                                  # Ethanol -> Propanol
    ]
    
    df = batch_analyze(model, example_pairs, DEVICE)
    
    print("Batch Analysis Results")
    print("=" * 60)
    display_cols = ['smiles_a', 'smiles_b', 'riemannian_distance', 
                   'warping_factor', 'predicted_difficulty']
    print(df[display_cols].to_string(index=False))

## 5. Interpreting Results <a name="5-interpreting-results"></a>

### Understanding the Metrics

In [ ]:
def interpret_results(results_df):
    """
    Provide interpretation guidelines for analysis results.
    """
    print("Results Interpretation Guide")
    print("=" * 50)
    
    # Summary statistics
    print("\n1. DISTANCE METRICS")
    print("-" * 30)
    print(f"   Mean Riemannian distance: {results_df['riemannian_distance'].mean():.4f}")
    print(f"   Mean Euclidean distance:  {results_df['euclidean_distance'].mean():.4f}")
    print(f"   Mean Warping factor:      {results_df['warping_factor'].mean():.2f}x")
    
    print("\n2. WARPING FACTOR INTERPRETATION")
    print("-" * 30)
    print("   ~1.0x: Transformation difficulty matches Euclidean intuition")
    print("   >1.5x: Transformation is harder than it appears")
    print("   <0.8x: Transformation is easier than it appears")
    
    print("\n3. DIFFICULTY RANKING")
    print("-" * 30)
    sorted_df = results_df.sort_values('predicted_difficulty', ascending=False)
    print("   Hardest transformations:")
    for _, row in sorted_df.head(3).iterrows():
        print(f"     {row['smiles_a'][:20]:20} -> {row['smiles_b'][:20]:20} (difficulty: {row['predicted_difficulty']:.3f})")
    
    print("\n   Easiest transformations:")
    for _, row in sorted_df.tail(3).iterrows():
        print(f"     {row['smiles_a'][:20]:20} -> {row['smiles_b'][:20]:20} (difficulty: {row['predicted_difficulty']:.3f})")
    
    print("\n4. RECOMMENDATIONS")
    print("-" * 30)
    
    high_difficulty = results_df[results_df['predicted_difficulty'] > results_df['predicted_difficulty'].median()]
    if len(high_difficulty) > 0:
        print(f"   {len(high_difficulty)} pairs above median difficulty:")
        print("     - Consider more lambda windows")
        print("     - Increase sampling time")
        print("     - Check for core hopping issues")
    
    return sorted_df


if 'df' in dir() and len(df) > 0:
    sorted_results = interpret_results(df)

### Visualize Metric Weights

In [ ]:
def visualize_metric_weights(model):
    """
    Visualize the learned metric weights.
    """
    weights = model.get_metric_weights().detach().cpu().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    ax1 = axes[0]
    ax1.bar(range(len(weights)), weights, color='steelblue', alpha=0.7)
    ax1.axhline(y=1.0, color='red', linestyle='--', label='Baseline (1.0)')
    ax1.set_xlabel('Embedding Dimension')
    ax1.set_ylabel('Metric Weight')
    ax1.set_title('Learned Metric Weights by Dimension')
    ax1.legend()
    
    # Histogram
    ax2 = axes[1]
    ax2.hist(weights, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(x=1.0, color='red', linestyle='--', label='Baseline (1.0)')
    ax2.axvline(x=weights.mean(), color='green', linestyle='-', label=f'Mean ({weights.mean():.2f})')
    ax2.set_xlabel('Metric Weight')
    ax2.set_ylabel('Count')
    ax2.set_title('Distribution of Metric Weights')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMetric Weight Statistics:")
    print(f"  Min:  {weights.min():.4f}")
    print(f"  Max:  {weights.max():.4f}")
    print(f"  Mean: {weights.mean():.4f}")
    print(f"  Std:  {weights.std():.4f}")
    
    # Most important dimensions
    top_dims = np.argsort(weights)[-5:][::-1]
    print(f"\n  Top 5 most important dimensions: {top_dims}")


visualize_metric_weights(model)

## 6. Generating Reports <a name="6-generating-reports"></a>

### Export Results

In [ ]:
def generate_report(results_df, output_dir='./reports'):
    """
    Generate a comprehensive analysis report.
    
    Args:
        results_df: DataFrame with analysis results
        output_dir: Directory to save reports
    
    Returns:
        Path to generated report
    """
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save CSV
    csv_path = os.path.join(output_dir, f'ntm_analysis_{timestamp}.csv')
    results_df.to_csv(csv_path, index=False)
    print(f"Results saved to: {csv_path}")
    
    # Generate HTML report
    html_path = os.path.join(output_dir, f'ntm_report_{timestamp}.html')
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>NTM Analysis Report</title>
        <style>
            body {{ font-family: 'IBM Plex Mono', monospace; margin: 40px; }}
            h1 {{ color: #2563eb; }}
            table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
            th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
            th {{ background-color: #2563eb; color: white; }}
            tr:nth-child(even) {{ background-color: #f2f2f2; }}
            .summary {{ background-color: #f0f9ff; padding: 20px; border-radius: 8px; margin: 20px 0; }}
        </style>
    </head>
    <body>
        <h1>Neural Thermodynamic Metric Analysis Report</h1>
        <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        
        <div class="summary">
            <h2>Summary Statistics</h2>
            <ul>
                <li>Total pairs analyzed: {len(results_df)}</li>
                <li>Mean Riemannian distance: {results_df['riemannian_distance'].mean():.4f}</li>
                <li>Mean warping factor: {results_df['warping_factor'].mean():.2f}x</li>
                <li>Predicted difficulty range: [{results_df['predicted_difficulty'].min():.4f}, {results_df['predicted_difficulty'].max():.4f}]</li>
            </ul>
        </div>
        
        <h2>Detailed Results</h2>
        {results_df.to_html(index=False, classes='results-table')}
        
        <h2>Recommendations</h2>
        <ul>
            <li>Pairs with warping factor > 1.5x may require additional lambda windows</li>
            <li>High predicted difficulty suggests longer equilibration times</li>
            <li>Consider running convergence checks for difficult transformations</li>
        </ul>
    </body>
    </html>
    """
    
    with open(html_path, 'w') as f:
        f.write(html_content)
    
    print(f"HTML report saved to: {html_path}")
    
    return {'csv': csv_path, 'html': html_path}


# Generate report
if 'df' in dir() and len(df) > 0:
    report_paths = generate_report(df, output_dir='../reports')

## Summary

### Complete Workflow

1. **Setup**: Import dependencies, configure device
2. **Model**: Load pre-trained or train new model
3. **Analysis**: Process molecular pairs
4. **Interpretation**: Understand distances, warping, difficulty
5. **Reporting**: Export results for downstream use

### Key Functions

| Function | Purpose |
|----------|--------|
| `mol_to_graph()` | SMILES → Graph representation |
| `analyze_pair()` | Single pair analysis |
| `batch_analyze()` | Multiple pairs analysis |
| `interpret_results()` | Statistical interpretation |
| `generate_report()` | Export CSV and HTML reports |

### Next Steps

1. **Train on your data**: Use actual RBFE uncertainties as labels
2. **Validate**: Compare predictions to experimental convergence
3. **Integrate**: Use predictions to prioritize FEP calculations
4. **Iterate**: Retrain as you gather more data